# CLV 구성요소·공유 상품 속성을 결합한 M5 — seed48 한 번

OOM·Drive I/O 수정판: 희소 가중치의 역전파에서 노드수×노드수 밀집 행렬 생성을 제거했고, 빈번한 진행기록과 매 epoch 체크포인트는 Colab 로컬에 둡니다. Drive에는 25 epoch마다 복구본과 최종 결과만 저장합니다. 모델·seed·배치·목적함수·파라미터는 유지하며 M3 학습도 끄지 않습니다. 수정 전 GPU 학습 완료 결과는 없습니다.

반복 탐색한 seed42를 피하고, 동일 입력·분할·300epoch의 기존 M1 원본이 확인되는 seed48을 사용합니다. seed48도 이전에 사용했으므로 독립적인 미노출 확증은 아닙니다.

- 새 학습: M5 1회만. M1·대조군은 새로 학습하지 않습니다.
- M2: 세부 상품군·학습 단가의 공유 표현 + N/V 조건부 구매이력 특성 변환. 양성 상품의 전체 이력 기여를 학습 때 제외합니다.
- M3: 거래횟수·금액지분·최근성 및 N/V를 입력으로 엣지 가중치를 공동학습합니다. 기존 고정 M3와 다른 가설입니다.
- M4: 기존 고객별 총 학습량 보존 M4-C(lambda=.5). 고CLV 고객의 총 학습량을 늘리는 방식이 아닙니다.
- 하나의 optimizer로 처음부터 함께 학습하며, 사전학습·동결·외부 재정렬은 없습니다.
- ID64·속성16(상품군8+가격8), eta=.1, epsilon=.1, kappa=.2, 300epoch 고정. 중간25epoch 평가는 진단만 하고 best epoch를 고르지 않습니다.
- Dunnhumby DAY1~683 학습/684~690 개발평가. 최종 마지막주 test·holdout은 열지 않습니다.

N은 반복거래율, V는 장바구니 평균금액이며 최근365일의 기존 정의를 변경하지 않습니다. V와 상품 단가의 상호작용은 검증할 가설이지 정의상 동일한 개념이 아닙니다. M1 비교만으로 CLV 귀속·독립축 기여·유의성을 주장할 수 없습니다. 학습형 그래프는 고정그래프보다 느릴 수 있습니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, subprocess, sys

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
BASELINE_JSON = Path('/content/drive/MyDrive/논문/data/results_v3_dunnhumby_clv_reliability_orthogonal_nv_seed48_replication_v1/reports/result.json')
if not BASELINE_JSON.is_file():
    raise FileNotFoundError(f'기존 seed48 M1 원본이 필요합니다. M1을 새로 학습하지 않습니다: {BASELINE_JSON}')

SOURCE_COMMIT = 'd17cbc551cd372d54e6467cedfb04c699e440f2f'
REPO = Path('/content/clv-m5-shared-attributes-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin',SOURCE_COMMIT], check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip() == SOURCE_COMMIT
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import torch
assert torch.cuda.is_available(), 'GPU 런타임(L4 권장)을 선택하세요. 아직 학습하지 않았습니다.'
import clv_m5_shared_attributes_seed48_screen as screen
cfg = screen.configure(baseline_json=str(BASELINE_JSON))
assert cfg.seeds == (48,)
print('GPU:', torch.cuda.get_device_name(0))
print('새 학습: seed48 M5 1회. 기준 M1:', BASELINE_JSON)

## 데이터·기준결과 검증

기존 M1의 입력 해시, 학습설정, 개발분할, 300epoch 완료 여부를 검사합니다. 불일치하면 학습을 시작하지 않습니다. 원래 N/V 계산, 신규상품 정답, 전체 카탈로그, 고객별 학습량 보존도 확인합니다.

In [ ]:
cfg, prepared = screen.prepare(cfg)

## M5 한 번 학습

optimizer·난수상태를 포함한 체크포인트는 매 epoch Colab 로컬에 저장하고, Drive 복구본은 25 epoch마다 저장합니다. 같은 런타임에서는 마지막 epoch부터, 새 런타임에서는 마지막 25-epoch 복구본부터 재개합니다. Drive heartbeat는 쓰지 않습니다. 완료된 M5는 재학습하지 않으며 판독점은300epoch 마지막점으로 고정합니다.

In [ ]:
result = screen.run(cfg, prepared)

## 전체 결과 비교·다운로드

전체·저/중/고CLV의 모든 지표와 @10·20·50, 불리한 지표를 포함합니다. 두 경제지표@10의 M1 우위와 여섯 정확도99% 보호는 탐색 신호일 뿐 최종 성공 판정이 아닙니다. 원본 ZIP 전체를 확인한 뒤 후속 실험 여부를 판단합니다. 속성 블록의 점수 비중은 CLV 기여율이 아닙니다.

In [ ]:
import pandas as pd
from IPython.display import display
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

comparison = result['comparison'].query('epoch == 300').copy()
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(comparison[['metric','m1','m5','difference','relative_percent']])
    display(result['diagnostics'].query('epoch == 300'))
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
print('한 시드 개발 결과이며 유의성·CLV 귀속·최종 성공을 주장하지 않습니다.')
zip_path = Path('/content/clv_m5_shared_attributes_seed48_results.zip')
with ZipFile(zip_path, 'w', ZIP_DEFLATED) as archive:
    for key, path in result['paths'].items():
        archive.write(path, Path(path).name)
files.download(str(zip_path))